# Training Data Penuh

Catatan: notebook ini adalah training awal di RTX 5060 Ti dan sekarang disimpan sebagai arsip. Semua training final (pencarian learning rate, seed 43 dan 44, dan uji layer yang dibekukan) dijalankan ulang di RTX 5070 Ti pada notebook 06, dan hasil test ada di notebook 07. Angka di notebook ini tidak dipakai di paper.

Notebook ini melatih RoBERTa (`roberta-base`) dan DistilBERT (`distilbert-base-cased`) dengan seluruh data latih, yaitu 799.459 pasangan parent dan comment. Setiap model dinilai di 100.276 data validasi.

Ada 3 jenis input yang dibandingkan:
- comment saja, model hanya membaca comment,
- early fusion, parent dan comment digabung menjadi satu input,
- late fusion, parent dan comment dibaca terpisah lalu hasilnya digabung.

Selain membandingkan model dan jenis input, notebook ini juga melihat pengaruh learning rate. Learning rate adalah seberapa besar model mengubah dirinya setiap kali belajar dari satu batch data. Nilai yang dicoba adalah 1e-5, 2e-5, 3e-5, dan 5e-5.

Pengaturan yang sama untuk semua run:

| Pengaturan | Nilai |
|---|---|
| Panjang maksimal input | 256 token |
| Batch size | 32 |
| Epoch | 3 |
| Warmup | 10% langkah pertama |
| Weight decay | 0,01 |
| Optimizer | AdamW |
| Presisi | BF16 |
| Seed | 42 |
| Pemilihan epoch terbaik | macro-F1 validasi |

Data test tidak dilihat di notebook ini. Prediksi data test disimpan terpisah di folder `test_sealed/` dan baru dibuka setelah konfigurasi final dipilih.

## 1. Setup

Kode training yang sama dengan `04_pilot` dipanggil dari `src/training.py`. Hasil disimpan ke folder `data/training_rtx5060ti/full/`.

In [1]:
import sys, importlib
from pathlib import Path

import pandas as pd
import torch
import transformers

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT / "src"))
import training as T
importlib.reload(T)
transformers.logging.set_verbosity_error()

DATA_DIR = ROOT / "data/tokenized_full"
OUT_DIR = ROOT / "data/training_rtx5060ti/full"
PUTARAN = [1, 2, 3]

free, total = torch.cuda.mem_get_info()
print("GPU:", torch.cuda.get_device_name(), f"| VRAM terpakai {(total - free) / 1e9:.1f} / {total / 1e9:.1f} GB")
print("Pemilihan epoch terbaik:", T.DEFAULTS["select_metric"])

d:\Falah\TA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA GeForce RTX 5060 Ti | VRAM terpakai 1.2 / 17.1 GB
Pemilihan epoch terbaik: macro_f1


#### Insight
GPU yang dipakai NVIDIA GeForce RTX 5060 Ti, dan sebelum training memorinya baru terpakai 1,2 GB untuk tampilan layar. Epoch terbaik dipilih memakai macro-F1, yaitu rata-rata F1 kelas sarkas dan kelas bukan sarkas, supaya model yang terlalu sering menebak "sarkas" tidak ikut terpilih.

## 2. Daftar Run

Run dibagi menjadi tiga putaran supaya hasil yang paling penting selesai lebih dulu:

| Putaran | Isi | Jumlah run |
|---|---|---:|
| 1 | 3 input × 2 model, learning rate 3e-5 | 6 |
| 2 | early fusion × 2 model, learning rate 1e-5, 2e-5, 5e-5 | 6 |
| 3 | comment saja dan late fusion × 2 model, learning rate 1e-5, 2e-5, 5e-5 | 12 |

Setelah ketiga putaran selesai, setiap kombinasi input dan model punya hasil untuk keempat learning rate.

In [2]:
queue = T.build_queue(PUTARAN)
T.run_status(queue, OUT_DIR)

,run_id,round,input,model,lr,seed,status
0,R1_comment_roberta_lr3e-05_s42,1,comment,roberta,0.00003,42,belum
1,R1_comment_distilbert_lr3e-05_s42,1,comment,distilbert,0.00003,42,belum
2,R1_early_roberta_lr3e-05_s42,1,early,roberta,0.00003,42,belum
3,R1_early_distilbert_lr3e-05_s42,1,early,distilbert,0.00003,42,belum
4,R1_late_roberta_lr3e-05_s42,1,late,roberta,0.00003,42,belum
5,R1_late_distilbert_lr3e-05_s42,1,late,distilbert,0.00003,42,belum
6,R2_early_roberta_lr1e-05_s42,2,early,roberta,0.00001,42,belum
7,R2_early_distilbert_lr1e-05_s42,2,early,distilbert,0.00001,42,belum
8,R2_early_roberta_lr2e-05_s42,2,early,roberta,0.00002,42,belum
9,R2_early_distilbert_lr2e-05_s42,2,early,distilbert,0.00002,42,belum


#### Insight
Ada 24 run dan semuanya masih berstatus "belum" saat training dimulai, jadi folder hasil masih kosong.

## 3. Training

Run dijalankan satu per satu. Di setiap epoch, model belajar dari seluruh data latih, lalu dinilai di data validasi. Setiap 10% epoch, loss dan learning rate dicatat ke `train_log.csv`. Setelah 3 epoch, model dari epoch dengan macro-F1 terbaik dipakai untuk menebak data test dan diukur kecepatannya.

Kalau training terputus, sel ini cukup dijalankan ulang. Run yang sudah selesai dilewati, dan run yang terputus dilanjutkan dari epoch terakhir yang sudah selesai.

In [3]:
for cfg in queue:
    T.run_experiment(cfg, DATA_DIR, OUT_DIR)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 527.61it/s]


[R1_comment_roberta_lr3e-05_s42] train 799,459 | val 100,276 | 24,984 langkah/epoch | parameter 124.6 juta (dilatih 124.6 juta)
  epoch 1 |  2,499/24,984 | loss 0.6010 | lr 1.00e-05 | 3.1 mnt, sisa ±27.5 mnt
  epoch 1 |  4,997/24,984 | loss 0.5378 | lr 2.00e-05 | 6.1 mnt, sisa ±24.3 mnt
  epoch 1 |  7,496/24,984 | loss 0.5318 | lr 3.00e-05 | 9.1 mnt, sisa ±21.2 mnt
  epoch 1 |  9,994/24,984 | loss 0.5199 | lr 2.89e-05 | 12.1 mnt, sisa ±18.1 mnt
  epoch 1 | 12,492/24,984 | loss 0.5114 | lr 2.78e-05 | 15.1 mnt, sisa ±15.1 mnt
  epoch 1 | 14,991/24,984 | loss 0.5051 | lr 2.67e-05 | 18.1 mnt, sisa ±12.1 mnt
  epoch 1 | 17,489/24,984 | loss 0.5007 | lr 2.56e-05 | 21.2 mnt, sisa ±9.1 mnt
  epoch 1 | 19,988/24,984 | loss 0.4949 | lr 2.44e-05 | 24.2 mnt, sisa ±6.1 mnt
  epoch 1 | 22,486/24,984 | loss 0.4902 | lr 2.33e-05 | 27.3 mnt, sisa ±3.0 mnt
  epoch 1 | 24,984/24,984 | loss 0.4899 | lr 2.22e-05 | 30.3 mnt, sisa ±0.0 mnt
  epoch 1 selesai | train 30.3 mnt | val loss 0.4779 | acc 0.7724 P 0

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 987.08it/s]


[R1_comment_distilbert_lr3e-05_s42] train 799,459 | val 100,276 | 24,984 langkah/epoch | parameter 65.8 juta (dilatih 65.8 juta)
  epoch 1 |  2,499/24,984 | loss 0.6103 | lr 1.00e-05 | 1.6 mnt, sisa ±14.5 mnt
  epoch 1 |  4,997/24,984 | loss 0.5517 | lr 2.00e-05 | 3.2 mnt, sisa ±12.9 mnt
  epoch 1 |  7,496/24,984 | loss 0.5367 | lr 3.00e-05 | 4.8 mnt, sisa ±11.2 mnt
  epoch 1 |  9,994/24,984 | loss 0.5245 | lr 2.89e-05 | 6.4 mnt, sisa ±9.6 mnt
  epoch 1 | 12,492/24,984 | loss 0.5139 | lr 2.78e-05 | 8.0 mnt, sisa ±8.0 mnt
  epoch 1 | 14,991/24,984 | loss 0.5133 | lr 2.67e-05 | 9.6 mnt, sisa ±6.4 mnt
  epoch 1 | 17,489/24,984 | loss 0.5057 | lr 2.56e-05 | 11.2 mnt, sisa ±4.8 mnt
  epoch 1 | 19,988/24,984 | loss 0.5003 | lr 2.44e-05 | 12.8 mnt, sisa ±3.2 mnt
  epoch 1 | 22,486/24,984 | loss 0.4980 | lr 2.33e-05 | 14.4 mnt, sisa ±1.6 mnt
  epoch 1 | 24,984/24,984 | loss 0.4968 | lr 2.22e-05 | 16.0 mnt, sisa ±0.0 mnt
  epoch 1 selesai | train 16.0 mnt | val loss 0.5032 | acc 0.7579 P 0.7489

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 516.67it/s]


...

[R3_late_distilbert_lr2e-05_s42] train 799,459 | val 100,276 | 24,984 langkah/epoch | parameter 67.6 juta (dilatih 67.6 juta)
  epoch 1 |  2,499/24,984 | loss 0.6292 | lr 6.67e-06 | 2.6 mnt, sisa ±23.5 mnt
  epoch 1 |  4,997/24,984 | loss 0.5525 | lr 1.33e-05 | 5.2 mnt, sisa ±20.9 mnt
  epoch 1 |  7,496/24,984 | loss 0.5336 | lr 2.00e-05 | 7.8 mnt, sisa ±18.3 mnt
  epoch 1 |  9,994/24,984 | loss 0.5235 | lr 1.93e-05 | 10.4 mnt, sisa ±15.7 mnt
  epoch 1 | 12,492/24,984 | loss 0.5090 | lr 1.85e-05 | 13.0 mnt, sisa ±13.0 mnt
  epoch 1 | 14,991/24,984 | loss 0.5012 | lr 1.78e-05 | 15.7 mnt, sisa ±10.4 mnt
  epoch 1 | 17,489/24,984 | loss 0.4970 | lr 1.70e-05 | 18.3 mnt, sisa ±7.8 mnt
  epoch 1 | 19,988/24,984 | loss 0.4949 | lr 1.63e-05 | 20.9 mnt, sisa ±5.2 mnt
  epoch 1 | 22,486/24,984 | loss 0.4912 | lr 1.56e-05 | 23.5 mnt, sisa ±2.6 mnt
  epoch 1 | 24,984/24,984 | loss 0.4881 | lr 1.48e-05 | 26.1 mnt, sisa ±0.0 mnt
  epoch 1 selesai | train 26.1 mnt | val loss 0.4812 | acc 0.7686 P 0.7

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 875.43it/s]


[R3_late_roberta_lr5e-05_s42] train 799,459 | val 100,276 | 24,984 langkah/epoch | parameter 127.0 juta (dilatih 127.0 juta)
  epoch 1 |  2,499/24,984 | loss 0.5937 | lr 1.67e-05 | 5.1 mnt, sisa ±45.7 mnt
  epoch 1 |  4,997/24,984 | loss 0.5394 | lr 3.33e-05 | 10.1 mnt, sisa ±40.5 mnt
  epoch 1 |  7,496/24,984 | loss 0.5322 | lr 5.00e-05 | 15.2 mnt, sisa ±35.4 mnt
  epoch 1 |  9,994/24,984 | loss 0.5327 | lr 4.81e-05 | 20.2 mnt, sisa ±30.3 mnt
  epoch 1 | 12,492/24,984 | loss 0.5212 | lr 4.63e-05 | 25.3 mnt, sisa ±25.3 mnt
  epoch 1 | 14,991/24,984 | loss 0.5125 | lr 4.44e-05 | 30.3 mnt, sisa ±20.2 mnt
  epoch 1 | 17,489/24,984 | loss 0.5119 | lr 4.26e-05 | 35.4 mnt, sisa ±15.2 mnt
  epoch 1 | 19,988/24,984 | loss 0.5048 | lr 4.07e-05 | 40.4 mnt, sisa ±10.1 mnt
  epoch 1 | 22,486/24,984 | loss 0.4990 | lr 3.89e-05 | 45.5 mnt, sisa ±5.1 mnt
  epoch 1 | 24,984/24,984 | loss 0.5054 | lr 3.70e-05 | 50.6 mnt, sisa ±0.0 mnt
  epoch 1 selesai | train 50.6 mnt | val loss 0.4919 | acc 0.7640 P 

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 809.40it/s]


[R3_late_distilbert_lr5e-05_s42] train 799,459 | val 100,276 | 24,984 langkah/epoch | parameter 67.6 juta (dilatih 67.6 juta)
  epoch 1 |  2,499/24,984 | loss 0.6069 | lr 1.67e-05 | 2.6 mnt, sisa ±23.6 mnt
  epoch 1 |  4,997/24,984 | loss 0.5469 | lr 3.33e-05 | 5.2 mnt, sisa ±20.9 mnt
  epoch 1 |  7,496/24,984 | loss 0.5345 | lr 5.00e-05 | 7.8 mnt, sisa ±18.3 mnt
  epoch 1 |  9,994/24,984 | loss 0.5294 | lr 4.81e-05 | 10.4 mnt, sisa ±15.6 mnt
  epoch 1 | 12,492/24,984 | loss 0.5158 | lr 4.63e-05 | 13.0 mnt, sisa ±13.0 mnt
  epoch 1 | 14,991/24,984 | loss 0.5092 | lr 4.44e-05 | 15.6 mnt, sisa ±10.4 mnt
  epoch 1 | 17,489/24,984 | loss 0.5052 | lr 4.26e-05 | 18.2 mnt, sisa ±7.8 mnt
  epoch 1 | 19,988/24,984 | loss 0.5031 | lr 4.07e-05 | 20.9 mnt, sisa ±5.2 mnt
  epoch 1 | 22,486/24,984 | loss 0.5001 | lr 3.89e-05 | 23.5 mnt, sisa ±2.6 mnt
  epoch 1 | 24,984/24,984 | loss 0.4964 | lr 3.70e-05 | 26.2 mnt, sisa ±0.0 mnt
  epoch 1 selesai | train 26.2 mnt | val loss 0.4922 | acc 0.7651 P 0.7

#### Insight
Semua 24 run selesai tanpa error. Training berjalan dari 7 Oktober sore sampai 9 Oktober pagi, sekitar 36 jam tanpa berhenti.

Satu epoch RoBERTa butuh sekitar 31 menit untuk comment saja, 33 menit untuk early fusion, dan 50 menit untuk late fusion. DistilBERT kira-kira setengahnya, yaitu 16, 17, dan 26 menit.

Di bagian awal setiap run, learning rate naik pelan-pelan selama warmup sampai mencapai nilai yang ditentukan, lalu turun sampai 0 di akhir epoch 3. Loss latih juga turun dari epoch ke epoch di semua run, jadi model memang belajar.

Ada dua pola yang terlihat dari log di atas:
1. Learning rate 5e-5 terlalu besar untuk RoBERTa. Pada RoBERTa comment saja dengan learning rate 5e-5, learning rate mencapai puncaknya (5e-5) di sekitar 30% epoch pertama. Setelah itu loss tidak lagi turun dengan baik. Loss sempat berada di 0,527, lalu menjelang akhir epoch 1 malah naik menjadi 0,573. Model seperti "terlempar" karena langkah belajarnya terlalu besar. Pada learning rate yang lebih kecil, loss terus turun dengan mulus.
2. Learning rate yang lebih besar membuat model lebih cepat overfitting. Contohnya DistilBERT early fusion dengan learning rate 5e-5, loss validasi turun di epoch 2 lalu naik lagi di epoch 3 (0,478 menjadi 0,535). Artinya model mulai menghafal data latih. Sebaliknya dengan learning rate 1e-5, epoch terbaik selalu epoch 3, jadi model masih bisa belajar sampai akhir.

## 4. Hasil

Tabel berisi hasil validasi di epoch terbaik untuk setiap run, beserta angka efisiensinya.

Keterangan kolom:
- `val_accuracy`, `val_precision`, `val_recall`, `val_f1`, `val_macro_f1`: nilai di data validasi,
- `train_min_total`: lama training 3 epoch dalam menit (tanpa waktu evaluasi),
- `peak_mem_gb`: memori GPU tertinggi saat training,
- `latency_bs1_ms`: waktu menebak 1 data dalam milidetik kalau data diproses satu per satu,
- `latency_bs32_ms`: waktu menebak per data kalau 32 data diproses sekaligus.

Selisih nilai di bawah ditulis dalam poin, misalnya 0,7962 dan 0,7834 berselisih 1,28 poin.

In [4]:
runs = pd.read_csv(OUT_DIR / "runs.csv")
cols = ["run_id", "best_epoch", "val_accuracy", "val_precision", "val_recall", "val_f1", "val_macro_f1",
        "train_min_total", "peak_mem_gb", "latency_bs1_ms", "latency_bs32_ms"]
display(runs[cols].round(4))

,run_id,best_epoch,val_accuracy,val_precision,val_recall,val_f1,val_macro_f1,train_min_total,peak_mem_gb,latency_bs1_ms,latency_bs32_ms
0,R1_comment_roberta_lr3e-05_s42,3,0.7808,0.7772,0.7796,0.7784,0.7808,92.0666,5.0840,15.9276,0.5641
1,R1_comment_distilbert_lr3e-05_s42,2,0.7669,0.7760,0.7423,0.7588,0.7667,48.1486,2.6316,7.7130,0.2793
2,R1_early_roberta_lr3e-05_s42,2,0.7927,0.7994,0.7747,0.7868,0.7926,100.0666,5.1063,15.3514,1.1613
3,R1_early_distilbert_lr3e-05_s42,2,0.7742,0.7654,0.7825,0.7738,0.7742,52.4548,2.6395,7.6691,0.6115
4,R1_late_roberta_lr3e-05_s42,3,0.7893,0.7782,0.8019,0.7899,0.7893,150.2840,8.3798,27.3933,1.5393
5,R1_late_distilbert_lr3e-05_s42,2,0.7756,0.7703,0.7772,0.7738,0.7755,77.5088,4.2835,13.3500,0.8019
6,R2_early_roberta_lr1e-05_s42,3,0.7962,0.7850,0.8087,0.7967,0.7962,99.5151,5.1063,15.3495,1.1636
7,R2_early_distilbert_lr1e-05_s42,3,0.7732,0.7714,0.7683,0.7699,0.7731,51.8788,2.6395,7.6400,0.6078
8,R2_early_roberta_lr2e-05_s42,2,0.7951,0.7972,0.7847,0.7909,0.7950,99.3562,5.1054,15.1696,1.1605
9,R2_early_distilbert_lr2e-05_s42,3,0.7745,0.7705,0.7737,0.7721,0.7744,51.9789,2.6395,7.4610,0.6067


#### Insight: pengaruh learning rate

Macro-F1 validasi untuk setiap learning rate:

| Input | Model | 1e-5 | 2e-5 | 3e-5 | 5e-5 |
|---|---|---:|---:|---:|---:|
| Comment saja | RoBERTa | 0,7827 | 0,7834 | 0,7808 | 0,7697 |
| Comment saja | DistilBERT | 0,7672 | 0,7677 | 0,7667 | 0,7658 |
| Early fusion | RoBERTa | 0,7962 | 0,7950 | 0,7926 | 0,7854 |
| Early fusion | DistilBERT | 0,7731 | 0,7744 | 0,7742 | 0,7715 |
| Late fusion | RoBERTa | 0,7918 | 0,7916 | 0,7893 | 0,7826 |
| Late fusion | DistilBERT | 0,7767 | 0,7759 | 0,7755 | 0,7707 |

RoBERTa lebih cocok dengan learning rate kecil. Learning rate 1e-5 dan 2e-5 hasilnya hampir sama dan selalu yang terbaik, sedangkan 5e-5 selalu paling rendah, sekitar 1 sampai 1,4 poin di bawah yang terbaik. Ini sesuai dengan loss yang naik lagi pada learning rate 5e-5 di bagian 3.

DistilBERT hampir tidak terpengaruh learning rate. Selisih antara hasil terbaik dan terburuk hanya 0,2 sampai 0,6 poin.

Learning rate 3e-5 yang dulu terpilih di pilot lama (data 40.000) ternyata bukan yang terbaik untuk data penuh, terutama untuk RoBERTa. Jadi learning rate yang cocok untuk data kecil belum tentu cocok untuk data besar.

#### Insight: hasil terbaik setiap konfigurasi

Learning rate terbaik dipilih dari macro-F1 validasi tertinggi:

| Input | Model | Learning rate | Accuracy | Precision | Recall | F1 | Macro-F1 |
|---|---|---|---:|---:|---:|---:|---:|
| Comment saja | RoBERTa | 2e-5 | 0,7834 | 0,7788 | 0,7840 | 0,7814 | 0,7834 |
| Comment saja | DistilBERT | 2e-5 | 0,7681 | 0,7820 | 0,7354 | 0,7580 | 0,7677 |
| Early fusion | RoBERTa | 1e-5 | 0,7962 | 0,7850 | 0,8087 | 0,7967 | 0,7962 |
| Early fusion | DistilBERT | 2e-5 | 0,7745 | 0,7705 | 0,7737 | 0,7721 | 0,7744 |
| Late fusion | RoBERTa | 1e-5 | 0,7918 | 0,7841 | 0,7982 | 0,7911 | 0,7918 |
| Late fusion | DistilBERT | 1e-5 | 0,7768 | 0,7782 | 0,7664 | 0,7723 | 0,7767 |

Hasil terbaik didapat RoBERTa dengan early fusion, yaitu F1 0,7967. Angka ini jauh di atas hasil percobaan sebelumnya dengan 40.000 data (F1 0,743). Kemungkinan karena data latih sekarang 20 kali lebih banyak, dan huruf kapital serta tanda baca tidak lagi dihapus.

Precision dan recall di semua konfigurasi cukup seimbang, sekitar 0,73 sampai 0,81, jadi model tidak berat sebelah ke salah satu kelas.

#### Insight: RoBERTa dibanding DistilBERT

Perbandingan pada learning rate 3e-5 (putaran 1), supaya kedua model dibandingkan dengan pengaturan yang sama:

| | Comment saja | Early fusion | Late fusion |
|---|---:|---:|---:|
| Waktu training 3 epoch (menit) RoBERTa / DistilBERT | 92 / 48 | 100 / 52 | 150 / 78 |
| DistilBERT lebih cepat training | 1,91× | 1,91× | 1,94× |
| Waktu menebak 1 data (ms) RoBERTa / DistilBERT | 15,9 / 7,7 | 15,4 / 7,7 | 27,4 / 13,4 |
| DistilBERT lebih cepat menebak | 2,07× | 2,00× | 2,05× |
| Memori (GB) RoBERTa / DistilBERT | 5,08 / 2,63 | 5,11 / 2,64 | 8,38 / 4,28 |

Dengan learning rate terbaik masing-masing, macro-F1 RoBERTa lebih tinggi 1,57 poin (comment saja), 2,18 poin (early fusion), dan 1,51 poin (late fusion) dibanding DistilBERT.

Jadi kalau memakai DistilBERT, performa turun sekitar 1,5 sampai 2,2 poin, tapi training dan prediksinya sekitar 2 kali lebih cepat dan memorinya sekitar setengah. Polanya sama di ketiga jenis input.

#### Insight: pengaruh konteks parent dan cara penggabungannya

Dengan learning rate terbaik masing-masing, menambahkan parent menaikkan macro-F1 dibanding comment saja:

| | RoBERTa | DistilBERT |
|---|---:|---:|
| Early fusion dibanding comment saja | +1,28 poin | +0,67 poin |
| Late fusion dibanding comment saja | +0,84 poin | +0,90 poin |

Pada data penuh, parent ternyata membantu, terutama untuk RoBERTa. Ini berbeda dengan percobaan sebelumnya dengan 40.000 data, yang selisihnya kurang dari 0,01. Kemungkinan dengan data yang lebih banyak, model bisa belajar memanfaatkan konteks.

Dari sisi waktu, early fusion hanya 1,09 kali lebih lama dari comment saja, padahal teksnya hampir tiga kali lebih panjang. Jadi menambahkan parent lewat early fusion hampir tidak menambah waktu training.

Late fusion tidak lebih baik dari early fusion. Pada RoBERTa hasilnya 0,44 poin lebih rendah, dan pada DistilBERT hanya 0,23 poin lebih tinggi. Padahal late fusion jauh lebih berat, yaitu training 1,5 kali lebih lama, menebak sekitar 1,75 kali lebih lambat, dan memori sekitar 1,6 kali lebih besar. Penyebabnya, late fusion menjalankan model dua kali, sekali untuk parent dan sekali untuk comment. Untuk data ini, early fusion adalah cara penggabungan yang lebih baik.

#### Catatan
Semua hasil ini berasal dari satu seed (42). Dengan 100.276 data validasi, selisih kurang dari sekitar 0,3 poin belum bisa dianggap berbeda, misalnya antara learning rate 1e-5 dan 2e-5. Selisih RoBERTa dan DistilBERT (1,5 sampai 2,2 poin) jauh lebih besar dari itu, tetapi tetap perlu dipastikan dengan seed lain.

## 5. Kesimpulan

1. Seluruh 24 run selesai dengan data penuh dalam sekitar 36 jam.
2. RoBERTa paling baik dengan learning rate 1e-5 atau 2e-5. Learning rate 5e-5 terlalu besar dan membuat training tidak stabil. DistilBERT hampir tidak terpengaruh learning rate.
3. Hasil terbaik adalah RoBERTa dengan early fusion (accuracy 0,7962, precision 0,7850, recall 0,8087, F1 0,7967).
4. DistilBERT sekitar 2 kali lebih cepat dan hemat memori, dengan performa 1,5 sampai 2,2 poin lebih rendah dari RoBERTa.
5. Konteks parent membantu, dan early fusion adalah cara penggabungan terbaik karena hasilnya setara atau lebih baik dari late fusion dengan biaya yang jauh lebih kecil.

Langkah berikutnya:
- menjalankan seed 43 dan 44 untuk konfigurasi terbaik setiap input dan model, supaya hasilnya bisa dilaporkan sebagai rata-rata dari 3 seed,
- menjalankan percobaan jumlah layer yang dilatih (early fusion),
- membuka data test untuk konfigurasi final.